# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Scenario Generation** - AI generates wilderness rescue scenario
2. **Simulation** - Student navigates scenario with human-in-the-loop
3. **Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [1]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import enable_tracing, simulation_session

# Initialize MLflow tracing
enable_tracing()

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 1: Scenario Generation

Generate a complete wilderness rescue scenario from minimal host configuration.

In [2]:
# Configuration
config = HostConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Generating: {config.activity_type} scenario for {config.num_participants} participants"
)
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Scenario ID: {scenario_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating: hiking scenario for 3 participants
------------------------------------------------------------
✓ Title: Alpine Exposure & Trauma: The Ridge Incident
✓ Setting: High Alpine ridge, 8000ft elevation, late autumn. Sudden weather change: temperature dropped rapidly to 40°F (4°C), wind gusting to 30mph with light freezing rain. Terrain is rocky and steep.
✓ Patient: Alex, 35. Alert and oriented x4. Shivering visibly. Cold, pale skin. Complaining of severe pain in the left ankle after a fall on a technical rock section. Unable to bear weight.
✓ Turns: 5
✓ Scenario ID: scn-d96af655

Learning Objectives:
  • Prioritizing life-threatening conditions (hypothermia) over orthopedic injuries
  • Conducting a systematic head-to-toe patient assessment in a wilderness setting
  • Effective splinting techniques for ankle injuries in a cold environment


Trace(trace_id=tr-6d47f9d6f93d895cc69328a4d4f37306)

### Display Scenario Structure

In [3]:
for turn in scenario.turns:
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"\n{'=' * 60}")
    print(f"Turn {turn.turn_id} {marker}")
    print(f"{'=' * 60}")
    print(
        turn.narrative_text[:200] + "..."
        if len(turn.narrative_text) > 200
        else turn.narrative_text
    )
    print("\nChoices:")
    for choice in turn.choices:
        mark = "✓" if choice.is_correct else " "
        next_turn = (
            "END" if choice.next_turn_id is None else f"Turn {choice.next_turn_id}"
        )
        print(f"  [{mark}] {choice.description[:60]}... → {next_turn}")


Turn 0 (START)
You and two hiking partners are traversing a rocky, exposed high alpine ridge when the weather turns suddenly. Temperatures have plummeted to 40°F with freezing rain and high winds. Alex, one of your ...

Choices:
  [✓] Immediately prioritize preventing further heat loss. Get Ale... → Turn 1
  [ ] Immediately focus on the ankle. Apply a rigid splint right o... → Turn 2

Turn 1 
You have successfully sheltered Alex from the wind and insulated them using your emergency gear. The shivering has begun to subside, and they are more alert. You must now address the ankle.

Choices:
  [✓] With the patient warm and dry, perform a thorough musculoske... → Turn 3
  [ ] Now that Alex is warmer, tell them to hop on one leg while l... → Turn 4

Turn 2 
By focusing only on the ankle and ignoring hypothermia, you have wasted precious time. The wind is increasing and the freezing rain is turning to sleet. Alex is becoming lethargic and less responsive....

Choices:
  [ ] Okay, let's keep

## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
    "debrief_report": None,
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [11]:
# Run simulation with MLflow session tracking
import mlflow

from summit_sim.tracing import log_simulation_results

with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    # Link traces to session using metadata
    mlflow.update_current_trace(
        metadata={
            "mlflow.trace.session": session_id,
            "mlflow.trace.user": "student",
            "scenario_id": scenario_id,
        }
    )

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    print("\n=== STARTING SIMULATION ===\n")

    while not simulation_complete:
        input_state = initial_state if turn_count == 0 else None
        turn_count += 1
        print(f"--- TURN {turn_count} ---")

        async for event in graph.astream(input_state, graph_config):
            if "__interrupt__" in event:
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n📖 {interrupt_data['narrative']}\n")
                print("Choices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                while True:
                    try:
                        sel = int(input("\nSelect: ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid")
                    except ValueError:
                        print("Enter number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check for graph completion
            if event == {}:
                simulation_complete = True
                break

        # Check if state indicates completion
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("Safety stop")
            break

    print("\n=== SIMULATION COMPLETE ===")

    # Log final results to MLflow parent run
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
        debrief_report=current_state["debrief_report"],
    )
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")

2026/03/24 08:18:36 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.
2026/03/24 08:18:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d791d89c0> was created in a different Context
2026/03/24 08:18:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d791d8b40> was created in a different Context



📋 Session ID: d6adb2c5-a418-4d62-8542-dfc735f6606b
🎮 Starting simulation...

INTERACTIVE SIMULATION


=== STARTING SIMULATION ===

--- TURN 1 ---

📖 You and two hiking partners are traversing a rocky, exposed high alpine ridge when the weather turns suddenly. Temperatures have plummeted to 40°F with freezing rain and high winds. Alex, one of your partners, slips on wet rock and falls. Alex is sitting on the trail, shivering violently, and clutching their left ankle, which is swollen and deformed. They are unable to stand. The wind is biting.

Choices:
  1. Immediately prioritize preventing further heat loss. Get Alex out of the wind, remove wet layers, and add insulation (sleeping bag/pad).
  2. Immediately focus on the ankle. Apply a rigid splint right over the top of their hiking boot to stabilize the suspected fracture, then worry about the cold.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/24 08:18:39 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d7930a2c0> was created in a different Context
2026/03/24 08:18:43 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d791da800> was created in a different Context
2026/03/24 08:18:43 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d7af9ae80> was created in a different Context
2026/03/24 08:18:43 WARNING mlflow.utils.autologging_utils: Encountered un

--- TURN 2 ---

📖 You have successfully sheltered Alex from the wind and insulated them using your emergency gear. The shivering has begun to subside, and they are more alert. You must now address the ankle.

Choices:
  1. With the patient warm and dry, perform a thorough musculoskeletal exam, check CMS, and apply a splint. Then, evaluate the feasibility of evacuation in the current weather.
  2. Now that Alex is warmer, tell them to hop on one leg while leaned on you and your partner to get back to the treeline as fast as possible, ignoring the ankle for now.


2026/03/24 08:18:46 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/24 08:18:46 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> a

--- TURN 3 ---

📖 You attempt to move the patient, but the exertion of hopping is taxing Alex's energy reserves and they are once again becoming cold. You realize your error. What is your final decision as they begin to stumble along the trail?

Choices:
  1. We have to get out now. It's too cold to stay. Alex manages to hop on one leg, but the exertion causes them to rapidly chill again. They collapse 400 yards down the trail. Scenario ends: Mismanagement of patient energy and exposure.
  2. We refuse to move because it's too dangerous due to the terrain and weather. We set up an emergency bivouac to survive the night. Scenario ends: Correct decision to prioritize safety over speed.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/24 08:19:00 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d7934f300> was created in a different Context
2026/03/24 08:19:03 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


=== SIMULATION COMPLETE ===

✓ Results logged to MLflow
✓ Total turns: 3
✓ Learning moments: 2
🏃 View run sim-demo-class-2024-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/6fe797a4f89f4447bbfe160a008f4d0f
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-ea9f77a219a8920bf3730212d29ddb61), Trace(trace_id=tr-24a2c0526fe3fdff1b2d24c340568f81)]

2026/03/24 08:19:10 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f9db7adb4c0> at 0x7f9d791ca480> was created in a different Context


## 4. Phase 3: Debrief

View the comprehensive performance report generated automatically by the simulation graph.

In [12]:
debrief_report = current_state["debrief_report"]

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report.final_score:.1f}%")
status_emoji = "✅" if debrief_report.completion_status == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report.completion_status.upper()}")

print("\n📝 Summary:")
print(debrief_report.summary)

if debrief_report.key_mistakes:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report.key_mistakes:
        print(f"  • {mistake}")

if debrief_report.strong_actions:
    print("\n💪 Strong Actions:")
    for action in debrief_report.strong_actions:
        print(f"  • {action}")

if debrief_report.teaching_points:
    print("\n📚 Teaching Points:")
    for point in debrief_report.teaching_points:
        print(f"  • {point}")

if debrief_report.best_next_actions:
    print("\n🎯 Recommendations:")
    for rec in debrief_report.best_next_actions:
        print(f"  • {rec}")

DEBRIEF REPORT

📊 Final Score: 66.7%
❌ Status: FAIL

📝 Summary:
The student demonstrated a strong grasp of fundamental wilderness priorities, specifically the need to stabilize hypothermic patients before addressing orthopedic injuries. However, the simulation revealed a gap in clinical judgment regarding patient transport. By proposing an unsafe movement technique, the student nearly escalated the patient’s injury and put the rescue team at risk. Ultimately, the student correctly prioritized safety by choosing to shelter in place. Given the 66.7% score, this performance falls just below the competency threshold.

⚠️ Key Mistakes:
  • Suggested an unsafe transport method (hopping) for a patient with an unstable lower-extremity injury.
  • Failed to fully assess the patient's injury severity before suggesting movement, risking further damage to the ankle.

💪 Strong Actions:
  • Correctly identified that hypothermia management is the immediate life-safety priority in alpine environments.

Trace(trace_id=tr-08f1705c950fef4cd0a6cc8eb0c30fe9)

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [13]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry["was_correct"] else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: You and two hiking partners are traversing a rocky, exposed high alpine ridge wh...
  Choice: Immediately prioritize preventing further heat loss. Get Alex out of the wind, remove wet layers, and add insulation (sleeping bag/pad).
  Feedback: Excellent decision. Prioritizing thermoregulation is critical here; hypothermia can impair Alex's co...

Turn 2 ✗
  Situation: You have successfully sheltered Alex from the wind and insulated them using your...
  Choice: Now that Alex is warmer, tell them to hop on one leg while leaned on you and your partner to get back to the treeline as fast as possible, ignoring the ankle for now.
  Feedback: While it is natural to want to descend to lower terrain, forcing an injured patient to hop through r...

Turn 3 ✓
  Situation: You attempt to move the patient, but the exertion of hopping is taxing Alex's en...
  Choice: We refuse to move because it's too dangerous due to the terrain and weather. We set up an em

In [14]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Hypothermia management: Protecting the patient from wind and moisture with an emergency bivouac is critical for survival.
2. Risk Assessment: Recognizing when movement risks the patient's condition more than waiting for search and rescue is a vital wilderness decision-making skill.


---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.